In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import spatialdata as sd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import senepy as sp
from typing import Optional, Tuple, Literal
import math



In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})


plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
output_dir = str(P.results.figures / "figure_6")
os.makedirs(output_dir, exist_ok=True)


In [ ]:
epi_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_20_50_harmony_batch_05_pt_05_ssg.h5ad"))


# setup

In [ ]:
epi_obs = epi_adata.obs[epi_adata.obs["annotation_final_coarse"] == "Epithelial"]

for col in ["tissue_type_cell_level", "tissue_type_dysplasia_cell_level"]:
    print(f"\n=== {col} ===")
    counts = epi_obs.groupby(["core_id", col]).size().reset_index(name="n_cells")
    core_label_counts = counts.groupby("core_id").filter(lambda x: len(x) > 1)  
    small_labels = core_label_counts[core_label_counts["n_cells"] < 10]
    if len(small_labels) > 0:
        print(f"WARNING: cores with a label < 10 cells:")
        print(small_labels.to_string(index=False))
    else:
        print("All labels in mixed cores have >= 10 cells.")

cell_ids = epi_adata.obs["cell_id"].astype(str)
n_total = len(cell_ids)
n_unique = cell_ids.nunique()
n_dupes = n_total - n_unique

if n_dupes > 0:
    dupe_ids = cell_ids[cell_ids.duplicated(keep=False)].unique().tolist()
    dupes = epi_adata.obs[cell_ids.isin(dupe_ids)]



In [ ]:
tissue_colors = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

colors_dutch = ['#F79F1F',
 '#1289A7',
 '#009432',
 '#9980FA',
 '#EA2027',
 '#833471',
 '#1B1464']

colors_dutch_y = ['#F79F1F',
 '#A3CB38',
 '#1289A7',
 '#9980FA',
 '#ED4C67',
 '#833471',
 '#D980FA',
 '#009432',
 '#0652DD',
 '#EA2027',
'#1B1464',
 '#EE5A24']

colors_dutch_long = [
    '#FFC312', '#C4E538', '#12CBC4', '#FDA7DF', '#ED4C67',
    '#F79F1F', '#A3CB38', '#1289A7', '#D980FA', '#B53471',
    '#EE5A24', '#009432', '#0652DD', '#9980FA', '#833471',
    '#EA2027', '#006266', '#1B1464', '#5758BB', '#6F1E51'
]

In [ ]:
import seaborn as sns

unique_patients = sorted(epi_adata.obs['patient_id'].astype(str).unique())
patient_colors_shared = dict(zip(unique_patients, sns.color_palette('tab20', len(unique_patients))))

# heatmaps

In [ ]:
def plot_score_distribution_heatmap(adata, score_col, n_bins=100, tissue_palette=None,
                                     patient_col='patient_id', tissue_col='tissue_type_cell_level_normal_split',
                                     save_path=None, ylabel=None):
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.colors as mcolors

    scores = adata.obs[score_col].values
    bin_edges = np.linspace(scores.min(), scores.max(), n_bins + 1)
    bin_labels = [f'{bin_edges[i]:.3f}' for i in range(n_bins)]

    df = pd.DataFrame({
        'patient': adata.obs[patient_col].astype(str).values,
        'tissue_type': adata.obs[tissue_col].astype(str).values,
        'score': scores,
    })

    df['score_bin'] = pd.cut(df['score'], bins=bin_edges, labels=bin_labels, include_lowest=True)

    counts = df.groupby(['patient', 'tissue_type', 'score_bin'], observed=False).size().reset_index(name='count')
    totals = df.groupby(['patient', 'tissue_type']).size().reset_index(name='total')
    counts = counts.merge(totals, on=['patient', 'tissue_type'])
    counts['pct'] = np.log1p(counts['count'] / counts['total'] * 100)

    counts['sample'] = counts['patient'].astype(str) + ' | ' + counts['tissue_type'].astype(str)
    heatmap_df = counts.pivot(index='sample', columns='score_bin', values='pct').fillna(0)

    mean_score = df.groupby(['patient', 'tissue_type'])['score'].median()
    mean_score.index = mean_score.index.map(lambda x: f'{x[0]} | {x[1]}')
    heatmap_df = heatmap_df.loc[mean_score.reindex(heatmap_df.index).sort_values().index]

    heatmap_df = heatmap_df.T
    heatmap_df = heatmap_df.iloc[::-1]

    sample_tissue = counts[['sample', 'tissue_type']].drop_duplicates()
    tissue_map = dict(zip(sample_tissue['sample'], sample_tissue['tissue_type']))
    unique_tissues = sorted(set(tissue_map.values()))

    if tissue_palette is None:
        tissue_palette = dict(zip(unique_tissues, sns.color_palette('Set2', len(unique_tissues))))

    unique_patients = sorted(set(df['patient']))
    patient_palette = dict(zip(unique_patients, sns.color_palette('tab20', len(unique_patients))))

    n_samples = len(heatmap_df.columns)
    fig_width = max(n_samples * 0.55 + 2, 24)

    fig = plt.figure(figsize=(fig_width, 16))
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 0.03, 0.03], hspace=0.02)

    ax_heat = fig.add_subplot(gs[0])
    ax_tissue = fig.add_subplot(gs[1], sharex=ax_heat)
    ax_patient = fig.add_subplot(gs[2], sharex=ax_heat)

    im = ax_heat.imshow(heatmap_df.values, aspect='auto', cmap='viridis',
                        interpolation='nearest')
    ax_heat.set_ylabel(ylabel if ylabel else f'{score_col} (low → high)',
                       fontsize=20, fontweight='bold', labelpad=15)

    ytick_positions = list(range(0, len(heatmap_df), 5))
    ax_heat.set_yticks(ytick_positions)
    ax_heat.set_yticklabels([heatmap_df.index[i] for i in ytick_positions], fontsize=14)
    ax_heat.set_xticks([])

    for i, s in enumerate(heatmap_df.columns):
        ax_tissue.bar(i, 1, width=1.0, color=mcolors.to_hex(tissue_palette[tissue_map[s]]),
                      edgecolor='none')
    ax_tissue.set_xlim(-0.5, len(heatmap_df.columns) - 0.5)
    ax_tissue.set_ylim(0, 1)
    ax_tissue.set_yticks([0.5])
    ax_tissue.set_yticklabels(['Tissue Type'], fontsize=16)
    ax_tissue.set_xticks([])
    ax_tissue.tick_params(length=0)
    for spine in ax_tissue.spines.values():
        spine.set_visible(False)

    for i, s in enumerate(heatmap_df.columns):
        ax_patient.bar(i, 1, width=1.0, color=mcolors.to_hex(patient_palette[s.split(' | ')[0]]),
                       edgecolor='none')
    ax_patient.set_xlim(-0.5, len(heatmap_df.columns) - 0.5)
    ax_patient.set_ylim(0, 1)
    ax_patient.set_yticks([0.5])
    ax_patient.set_yticklabels(['Patient'], fontsize=16)
    ax_patient.set_xticks([])
    ax_patient.tick_params(length=0)
    for spine in ax_patient.spines.values():
        spine.set_visible(False)

    fig.subplots_adjust(left=0.15, right=0.75, top=0.95, bottom=0.05)

    heatmap_pos = ax_heat.get_position()
    right_edge = heatmap_pos.x1
    top_edge = heatmap_pos.y1
    bottom_edge = heatmap_pos.y0
    mid_y = (top_edge + bottom_edge) / 2

    cbar_width = 0.012
    cbar_height = heatmap_pos.height * 0.4
    cbar_x = right_edge + 0.02
    cbar_y = mid_y - cbar_height / 2
    cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical')
    cbar.set_label('log(% of cells + 1)', fontsize=12)
    cbar.ax.tick_params(labelsize=10)

    legend_x = cbar_x + cbar_width + 0.04

    tissue_handles = [plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=tissue_palette[t],
                      markersize=12, label=t) for t in unique_tissues]
    fig.legend(handles=tissue_handles, title='Tissue Type',
               bbox_to_anchor=(legend_x, mid_y + 0.1), loc='lower left',
               frameon=True, fontsize=14, title_fontsize=15)

    patient_handles = [plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=patient_palette[p],
                       markersize=12, label=p) for p in unique_patients]
    fig.legend(handles=patient_handles, title='Patient',
               bbox_to_anchor=(legend_x, mid_y + 0.08), loc='upper left',
               frameon=True, fontsize=14, title_fontsize=15)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_score_distribution_heatmap(epi_adata, 'stemness_score', n_bins=50, tissue_palette=tissue_colors_normal_split,
                                save_path=os.path.join(output_dir, "stemness_score_hist_heatmap.pdf"))

In [ ]:
plot_score_distribution_heatmap(epi_adata, 'senepy_intestine_epi_0', n_bins=50, tissue_palette=tissue_colors_normal_split,
                                save_path=os.path.join(output_dir, "senepy_score_hist_heatmap.pdf"), ylabel='SenePy Score (low → high)')

# tissue type comparisons

In [ ]:
tissue_colors = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#3dcd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

colors_dutch = ['#F79F1F',
 '#1289A7',
 '#009432',
 '#9980FA',
 '#EA2027',
 '#833471',
 '#1B1464']

colors_dutch_y = ['#F79F1F',
 '#A3CB38',
 '#1289A7',
 '#9980FA',
 '#ED4C67',
 '#833471',
 '#D980FA',
 '#009432',
 '#0652DD',
 '#EA2027',
'#1B1464',
 '#EE5A24']

colors_dutch_long = [
    '#FFC312', '#C4E538', '#12CBC4', '#FDA7DF', '#ED4C67',
    '#F79F1F', '#A3CB38', '#1289A7', '#D980FA', '#B53471',
    '#EE5A24', '#009432', '#0652DD', '#9980FA', '#833471',
    '#EA2027', '#006266', '#1B1464', '#5758BB', '#6F1E51'
]

In [ ]:
epi_adata.obs['tissue_type_cell_level_normal_split'] = pd.Categorical(
    epi_adata.obs['tissue_type_cell_level_normal_split'],
    categories=['Dist_N', 'Adj_N', 'AD', 'CA'],
    ordered=True,
)

In [ ]:
stemness_genes_lau = ["CDX2","OLFM4","CD44","AXIN2","RNF43","EPHB2","LGR5","ASCL2"]


In [ ]:
order= ["Dist_N","Adj_N","AD","CA"]
stemness_genes_lau = ["CDX2","OLFM4","CD44","AXIN2","RNF43","EPHB2","LGR5","ASCL2"]
dp = sc.pl.dotplot(epi_adata, stemness_genes_lau, groupby="tissue_type_cell_level_normal_split",
                   title="", standard_scale="var", swap_axes=True, return_fig=True, categories_order=order)
ax_dict = dp.show(return_axes=True)
ax_dict["mainplot_ax"].tick_params(axis="both", labelsize=14)
ax_dict["mainplot_ax"].set_xticklabels(
    ax_dict["mainplot_ax"].get_xticklabels(), rotation=45, ha="right", fontsize=12
)
plt.savefig(os.path.join(output_dir, "stem_markers_by_tissue_type_dotplot_lau.pdf"), dpi=300, bbox_inches="tight")

plt.show()

# morans' spatial analysis

### setup

In [ ]:

ADATA_PATH  = str(P.processed.adata.all_cells / P.fn.all_cells_morans)
SDATA_DIR   = str(P.interim.sdata)

In [ ]:
all_cells_adata = sc.read_h5ad(ADATA_PATH)

before = len(all_cells_adata)
all_cells_adata = all_cells_adata[~all_cells_adata.obs['cell_id'].duplicated(keep=False)].copy()
print(f"{before} → {len(all_cells_adata)} cells ({before - len(all_cells_adata)} removed)")

In [ ]:
def make_spatialdata_dict(spatialdata_dir):
    """Load all filtered/normalized SpatialData .zarr files into a dictionary."""
    spatialdata_dict = {}
    entries = [f for f in os.listdir(spatialdata_dir) if f.startswith("fil")]
    for filename in sorted(entries):
        sdata = sd.read_zarr(f"{spatialdata_dir}/{filename}")
        key = filename.replace('filtered_normalized_xenium_', '').replace('.zarr', '')
        spatialdata_dict[key] = sdata
    return spatialdata_dict


def transfer_obs_columns(spatialdata_dict, reference_adata, columns):
    """Transfer annotation columns from a reference AnnData to SpatialData tables (join on cell_id)."""
    ref_obs = reference_adata.obs.set_index('cell_id')[columns]
    for section_key, sdata in spatialdata_dict.items():
        table = sdata.tables['table']
        for col in columns:
            if col in ref_obs.columns:
                table.obs[col] = table.obs['cell_id'].map(ref_obs[col])
    return spatialdata_dict


spatialdata_dict = make_spatialdata_dict(SDATA_DIR)
print(f"Loaded {len(spatialdata_dict)} SpatialData sections")





MIN_EPI_CELLS        = 100       

epi_obs = all_cells_adata.obs[all_cells_adata.obs['annotation_final_fine_cd8'] == 'Epithelial']
epi_counts = epi_obs.groupby(['batch', 'core_id']).size()
valid_pairs = epi_counts[epi_counts >= MIN_EPI_CELLS].reset_index()
valid_section_cores = set(zip(valid_pairs['batch'], valid_pairs['core_id']))
print(f"Valid (section, core) pairs: {len(valid_section_cores)}")


valid_cell_ids = set(all_cells_adata.obs['cell_id'].unique())

filtered_spatialdata_dict = {}
for section_key, sdata in spatialdata_dict.items():
    table = sdata.tables['table']
    valid_cores_for_section = {
        core for (batch, core) in valid_section_cores if batch == section_key
    }
    mask = (
        table.obs['cell_id'].isin(valid_cell_ids)
        & table.obs['core_id'].isin(valid_cores_for_section)
    )
    if mask.sum() > 0:
        sdata.tables['table'] = table[mask].copy()
        filtered_spatialdata_dict[section_key] = sdata
        print(f"  {section_key}: {mask.sum()} cells, "
              f"{sdata.tables['table'].obs['core_id'].nunique()} cores")
    else:
        print(f"  {section_key}: dropped (no valid cells)")


core_to_sections = {}
for section_key, sdata in filtered_spatialdata_dict.items():
    for core in sdata.tables['table'].obs['core_id'].unique():
        core_to_sections.setdefault(core, []).append(section_key)
duplicated = {c: s for c, s in core_to_sections.items() if len(s) > 1}
print(f"\nCores in multiple sections: {len(duplicated)}")
for core, sections in sorted(duplicated.items()):
    print(f"  {core}: {sections}")


TRANSFER_COLUMNS = [
    'annotation_final_fine', 'annotation_final_coarse',
    'tissue_type_cell_level', 'tissue_type_dysplasia_cell_level',
    'mixed_core_tissue_type', 'mixed_core_dysplasia',
    'tissue_type_cell_level_normal_split',
    'tissue_type_dysplasia_cell_level_normal_split',
    'epithelial_0.8',
    'annotation_final_fine_cd8',
    'GDF15_more_than_1_transcript', 'senepy_intestine_epi_0',
    'stemness_score', 'senepy_high', 'core_id', 'patient_id','GDF15_dense',
 'GDF15_morans_quad',
 'GDF15_morans_pval',
 'GDF15_morans_Ii',
 'stemness_score_dense',
 'stemness_score_morans_quad',
 'stemness_score_morans_pval',
 'stemness_score_morans_Ii',
 'senepy_intestine_epi_0_dense',
 'senepy_intestine_epi_0_morans_quad',
 'senepy_intestine_epi_0_morans_pval',
 'senepy_intestine_epi_0_morans_Ii'
]

filtered_spatialdata_dict = transfer_obs_columns(
    filtered_spatialdata_dict, all_cells_adata, TRANSFER_COLUMNS
)


for section_key, sdata in filtered_spatialdata_dict.items():
    table = sdata.tables['table']
    n_epi = (table.obs['annotation_final_fine_cd8'] == 'Epithelial').sum()
    n_total = len(table)
    print(f"{section_key}: {n_total} total, {n_epi} epithelial ({100*n_epi/n_total:.1f}%)")

## visualize overlaps

In [ ]:
DENSE_FEATURES = [
    "GDF15_dense",
    "stemness_score_dense",
    "senepy_intestine_epi_0_dense",
]

FEATURE_LABELS = {
    "GDF15_dense":                  "GDF15",
    "stemness_score_dense":         "Stemness",
    "senepy_intestine_epi_0_dense": "SenePy",
}

FEATURE_COLORS = {
    "GDF15_dense":                  "#d62728",
    "stemness_score_dense":         "#1f78b4",
    "senepy_intestine_epi_0_dense": "#f0c929",
}

OVERLAP_PAIRS = [
    ("GDF15_dense", "stemness_score_dense",                   "#7b2d8e"),
    ("stemness_score_dense", "senepy_intestine_epi_0_dense",  "#2ca02c"),
]

TRIPLE_COLOR    = "#000000"
COLOR_OTHER_EPI = "#e0e0e0"
COLOR_NON_EPI   = "#f5f5f5"
ALPHA_DENSE     = 0.9
ALPHA_OTHER_EPI = 0.4
ALPHA_NON_EPI   = 0.15
ALPHA_BOTH      = 1.0
ALPHA_ONE_OF    = 0.7
DOT_SIZE_DENSE  = 8
DOT_SIZE_OTHER  = 2
DOT_SIZE_NON_EPI = 0.5
DOT_SIZE_BOTH   = 12
DOT_SIZE_ONE_OF = 6
DOT_SIZE_TRIPLE = 14
FIGSIZE_PER_SUBPLOT = (4.5, 4.5)

### normal cores 

In [ ]:
sc.settings._vector_friendly = True

def plot_normal_cores_combined(
    filtered_spatialdata_dict,
    core_ids,
    tissue_col="tissue_type_cell_level_normal_split",
    feat_stem="stemness_score_dense",
    feat_sene="senepy_intestine_epi_0_dense",
    points_key="transcripts",
    gene_col="feature_name",
    transcript_gene="GDF15",
    color_stem="#44c3e3",
    color_sene="#F79F1F",
    color_other_epi="#d3d3d3",
    color_non_epi="#fafafa",
    color_transcript="#EA2027",
    dot_size_dense=12,
    dot_size_other=4,
    dot_size_non_epi=2,
    dot_size_transcript=6,
    alpha_dense=1.0,
    alpha_other_epi=0.6,
    alpha_non_epi=0.3,
    alpha_transcript=1.0,
    figsize_per_subplot=(4, 4),
    save_path=None,
):
    nrows = len(core_ids)
    ncols = 2
    fig, all_axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_subplot[0] * ncols, figsize_per_subplot[1] * nrows),
        squeeze=False,
    )

    def _clean_ax(ax):
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for row_idx, core_id in enumerate(core_ids):
        sdata_match = None
        core_obs = None
        section_key = None
        for sk, sdata in filtered_spatialdata_dict.items():
            table = sdata.tables['table']
            mask = table.obs['core_id'] == core_id
            if mask.any():
                sdata_match = sdata
                core_obs = table.obs[mask].copy()
                section_key = sk
                break

        if core_obs is None:
            print(f"Core {core_id} not found")
            for ax in all_axes[row_idx]:
                ax.set_visible(False)
            continue

        cell_shapes = sdata_match.shapes['cell_boundaries']
        core_spatial = cell_shapes[cell_shapes.index.isin(core_obs['cell_id'].values)]
        coord_map = {
            i: (g.centroid.x, g.centroid.y)
            for i, g in zip(core_spatial.index, core_spatial.geometry)
        }
        core_obs['cx'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan,))[0])
        core_obs['cy'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan, np.nan))[1])
        core_obs = core_obs.dropna(subset=['cx', 'cy'])

        tx_x, tx_y = np.array([]), np.array([])
        if points_key in sdata_match.points:
            import dask.dataframe as dd
            pts = sdata_match.points[points_key]
            if isinstance(pts, dd.DataFrame):
                pts = pts.compute()
            gdf15_pts = pts[pts[gene_col] == transcript_gene]
            xmin, ymin = core_obs['cx'].min(), core_obs['cy'].min()
            xmax, ymax = core_obs['cx'].max(), core_obs['cy'].max()
            pad = 50
            gdf15_pts = gdf15_pts[
                (gdf15_pts['x'] >= xmin - pad) & (gdf15_pts['x'] <= xmax + pad) &
                (gdf15_pts['y'] >= ymin - pad) & (gdf15_pts['y'] <= ymax + pad)
            ]
            tx_x = gdf15_pts['x'].values
            tx_y = gdf15_pts['y'].values
            print(f"  Core {core_id}: {len(tx_x)} GDF15 transcripts")
        else:
            print(f"  Warning: '{points_key}' not found for {section_key}")

        is_epi = core_obs['annotation_final_fine'] == 'Epithelial'
        is_non_epi = ~is_epi
        has_stem = (core_obs[feat_stem] == True) if feat_stem in core_obs.columns else pd.Series(False, index=core_obs.index)
        has_sene = (core_obs[feat_sene] == True) if feat_sene in core_obs.columns else pd.Series(False, index=core_obs.index)

        def _draw_bg(ax):
            if is_non_epi.any():
                ax.scatter(
                    core_obs.loc[is_non_epi, 'cx'], core_obs.loc[is_non_epi, 'cy'],
                    s=dot_size_non_epi, c=color_non_epi, alpha=alpha_non_epi,
                    linewidths=0, zorder=2, rasterized=True,
                )

        def _draw_transcripts(ax):
            if len(tx_x) > 0:
                ax.scatter(
                    tx_x, tx_y,
                    s=dot_size_transcript, c=color_transcript, alpha=alpha_transcript,
                    marker='x', linewidths=0.5, zorder=10,
                )

        axes = all_axes[row_idx]

        # Panel 0: Stemness
        ax = axes[0]
        _draw_bg(ax)
        is_dense = is_epi & has_stem
        is_other = is_epi & ~is_dense
        if is_other.any():
            ax.scatter(core_obs.loc[is_other, 'cx'], core_obs.loc[is_other, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if is_dense.any():
            ax.scatter(core_obs.loc[is_dense, 'cx'], core_obs.loc[is_dense, 'cy'],
                       s=dot_size_dense, c=color_stem, alpha=alpha_dense,
                       linewidths=0, zorder=5)
        _draw_transcripts(ax)
        if row_idx == 0:
            ax.set_title("Stemness", fontsize=11, fontweight='bold', color=color_stem)
        _clean_ax(ax)

        # Panel 1: SenePy
        ax = axes[1]
        _draw_bg(ax)
        is_dense = is_epi & has_sene
        is_other = is_epi & ~is_dense
        if is_other.any():
            ax.scatter(core_obs.loc[is_other, 'cx'], core_obs.loc[is_other, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if is_dense.any():
            ax.scatter(core_obs.loc[is_dense, 'cx'], core_obs.loc[is_dense, 'cy'],
                       s=dot_size_dense, c=color_sene, alpha=alpha_dense,
                       linewidths=0, zorder=5)
        _draw_transcripts(ax)
        if row_idx == 0:
            ax.set_title("SenePy", fontsize=11, fontweight='bold', color=color_sene)
        _clean_ax(ax)

        # Row label
        tissue = core_obs.get(tissue_col, pd.Series(dtype=str)).dropna().unique()
        tissue_str = tissue[0] if len(tissue) > 0 else "?"
        axes[0].set_ylabel(f"{core_id}\n[{tissue_str}]", fontsize=10, fontweight='bold', rotation=0, labelpad=60, va='center')

    fig.tight_layout()

    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"Saved to {save_path}")

    plt.show()


plot_normal_cores_combined(
    filtered_spatialdata_dict,
    core_ids=["N1-N-2", "L2-N-1", "P1-N-2"],
    save_path=str(P.results.figures / "figure_3" / "normal_cores_stem_sene_gdf15.pdf"),
)

sc.settings._vector_friendly = False

### back to other stuff

In [ ]:
DENSE_FEATURES = [
    "GDF15_dense",
    "stemness_score_dense",
    "senepy_intestine_epi_0_dense",
]

FEATURE_LABELS = {
    "GDF15_dense":                  "GDF15",
    "stemness_score_dense":         "Stemness",
    "senepy_intestine_epi_0_dense": "SenePy",
}

FEATURE_COLORS = {
    "GDF15_dense":                  "#d62728",
    "stemness_score_dense":         "#1f78b4",
    "senepy_intestine_epi_0_dense": "#f0c929",
}

OVERLAP_PAIRS = [
    ("GDF15_dense", "stemness_score_dense",                   "#7b2d8e"),
    ("stemness_score_dense", "senepy_intestine_epi_0_dense",  "#2ca02c"),
]

TRIPLE_COLOR    = "#000000"
COLOR_OTHER_EPI = "#e0e0e0"
COLOR_NON_EPI   = "#f5f5f5"
ALPHA_DENSE     = 0.9
ALPHA_OTHER_EPI = 0.4
ALPHA_NON_EPI   = 0.15
ALPHA_BOTH      = 1.0
ALPHA_ONE_OF    = 0.7
DOT_SIZE_DENSE  = 8
DOT_SIZE_OTHER  = 2
DOT_SIZE_NON_EPI = 0.5
DOT_SIZE_BOTH   = 12
DOT_SIZE_ONE_OF = 6
DOT_SIZE_TRIPLE = 14
FIGSIZE_PER_SUBPLOT = (4.5, 4.5)

TISSUE_ORDER = ['Dist_N', 'Adj_N', 'AD', 'CA']


CATEGORY_ORDER = [
    "GDF15 only",
    "Stemness only",
    "SenePy only",
    "GDF15 ∩ Stemness",
    "Stemness ∩ SenePy",
    "GDF15 ∩ SenePy",
    "All three",
]


CATEGORY_COLORS = {
    "GDF15 only":         "#EA2027",
    "Stemness only":      "#44c3e3",
    "SenePy only":        "#F79F1F",
    "GDF15 ∩ Stemness":   "#5758BB",
    "Stemness ∩ SenePy":  "#046e27",
    "GDF15 ∩ SenePy":     "#EE5A24",  # orange for the pair not shown spatially
    "All three":          "#000000",
}

In [ ]:
def plot_dense_grid_core_with_he(
    filtered_spatialdata_dict,
    core_id,
    dense_features=DENSE_FEATURES,
    feature_colors=FEATURE_COLORS,
    feature_labels=FEATURE_LABELS,
    overlap_pairs=OVERLAP_PAIRS,
    triple_color=TRIPLE_COLOR,
    color_other_epi=COLOR_OTHER_EPI,
    color_non_epi=COLOR_NON_EPI,
    color_any_one="#aaaaaa",
    alpha_dense=ALPHA_DENSE,
    alpha_other_epi=ALPHA_OTHER_EPI,
    alpha_non_epi=ALPHA_NON_EPI,
    alpha_both=ALPHA_BOTH,
    alpha_one_of=ALPHA_ONE_OF,
    dot_size_dense=DOT_SIZE_DENSE,
    dot_size_other=DOT_SIZE_OTHER,
    dot_size_non_epi=DOT_SIZE_NON_EPI,
    dot_size_both=DOT_SIZE_BOTH,
    dot_size_one_of=DOT_SIZE_ONE_OF,
    dot_size_triple=DOT_SIZE_TRIPLE,
    figsize_per_subplot=FIGSIZE_PER_SUBPLOT,
    category_colors=CATEGORY_COLORS,
    save_dir=None,
):
    sdata = None
    for sk, sd in filtered_spatialdata_dict.items():
        table = sd.tables['table']
        if core_id in table.obs['core_id'].values:
            sdata = sd
            break

    if sdata is None:
        print(f"Core {core_id} not found")
        return

    table = sdata.tables['table']
    core_obs = table.obs[table.obs['core_id'] == core_id].copy()

    cell_shapes = sdata.shapes['cell_boundaries']
    core_spatial = cell_shapes[cell_shapes.index.isin(core_obs['cell_id'].values)]
    coord_map = {
        i: (g.centroid.x, g.centroid.y)
        for i, g in zip(core_spatial.index, core_spatial.geometry)
    }
    core_obs['cx'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan,))[0])
    core_obs['cy'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan, np.nan))[1])
    core_obs = core_obs.dropna(subset=['cx', 'cy'])

    is_epi = core_obs['annotation_final_fine'] == 'Epithelial'
    is_non_epi = ~is_epi

    feat_a_col, feat_b_col, feat_c_col = dense_features
    has_g = (core_obs[feat_a_col] == True) if feat_a_col in core_obs.columns else pd.Series(False, index=core_obs.index)
    has_s = (core_obs[feat_b_col] == True) if feat_b_col in core_obs.columns else pd.Series(False, index=core_obs.index)
    has_p = (core_obs[feat_c_col] == True) if feat_c_col in core_obs.columns else pd.Series(False, index=core_obs.index)

    fig, axes = plt.subplots(
        2, 4,
        figsize=(figsize_per_subplot[0] * 4, figsize_per_subplot[1] * 2),
        squeeze=False,
    )

    def _clean_ax(ax):
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    def _draw_bg(ax):
        if is_non_epi.any():
            ax.scatter(
                core_obs.loc[is_non_epi, 'cx'], core_obs.loc[is_non_epi, 'cy'],
                s=dot_size_non_epi, c=color_non_epi, alpha=alpha_non_epi,
                linewidths=0, zorder=2, rasterized=True,
            )

    # Top-left: empty placeholder for H&E
    axes[0][0].set_facecolor('white')
    for spine in axes[0][0].spines.values():
        spine.set_visible(False)
    axes[0][0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    axes[0][0].set_title('H&E', fontsize=11, fontweight='bold', color='black')

    for col_idx, feat_col in enumerate(dense_features):
        ax = axes[0][col_idx + 1]
        fc = feature_colors[feat_col]
        label = feature_labels.get(feat_col, feat_col.replace("_dense", ""))

        _draw_bg(ax)

        is_dense = is_epi & (core_obs[feat_col] == True) if feat_col in core_obs.columns else is_epi & False
        is_other = is_epi & ~is_dense

        if is_other.any():
            ax.scatter(core_obs.loc[is_other, 'cx'], core_obs.loc[is_other, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if is_dense.any():
            ax.scatter(core_obs.loc[is_dense, 'cx'], core_obs.loc[is_dense, 'cy'],
                       s=dot_size_dense, c=fc, alpha=alpha_dense, linewidths=0, zorder=5, rasterized=True)

        ax.set_title(label, fontsize=11, fontweight='bold', color='black')
        _clean_ax(ax)

    for ovl_idx, (fa, fb, both_color) in enumerate(overlap_pairs):
        ax = axes[1][ovl_idx]
        ca = feature_colors[fa]
        cb = feature_colors[fb]
        la = feature_labels.get(fa, fa.replace("_dense", ""))
        lb = feature_labels.get(fb, fb.replace("_dense", ""))

        _draw_bg(ax)

        ha = (core_obs[fa] == True) if fa in core_obs.columns else pd.Series(False, index=core_obs.index)
        hb = (core_obs[fb] == True) if fb in core_obs.columns else pd.Series(False, index=core_obs.index)

        both    = is_epi & ha & hb
        only_a  = is_epi & ha & ~hb
        only_b  = is_epi & ~ha & hb
        neither = is_epi & ~ha & ~hb

        if neither.any():
            ax.scatter(core_obs.loc[neither, 'cx'], core_obs.loc[neither, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if only_a.any():
            ax.scatter(core_obs.loc[only_a, 'cx'], core_obs.loc[only_a, 'cy'],
                       s=dot_size_one_of, c=ca, alpha=alpha_one_of, linewidths=0, zorder=4, rasterized=True)
        if only_b.any():
            ax.scatter(core_obs.loc[only_b, 'cx'], core_obs.loc[only_b, 'cy'],
                       s=dot_size_one_of, c=cb, alpha=alpha_one_of, linewidths=0, zorder=4, rasterized=True)
        if both.any():
            ax.scatter(core_obs.loc[both, 'cx'], core_obs.loc[both, 'cy'],
                       s=dot_size_both, c=both_color, alpha=alpha_both, linewidths=0, zorder=6, rasterized=True)

        ax.set_title(f"{la} ∩ {lb}", fontsize=11, fontweight='bold', color='black')
        _clean_ax(ax)

    # Bottom row col 2: triple overlap
    ax = axes[1][2]
    _draw_bg(ax)

    triple  = is_epi & has_g & has_s & has_p
    any_one = is_epi & (has_g | has_s | has_p) & ~triple
    none_   = is_epi & ~has_g & ~has_s & ~has_p

    if none_.any():
        ax.scatter(core_obs.loc[none_, 'cx'], core_obs.loc[none_, 'cy'],
                   s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                   linewidths=0, zorder=3, rasterized=True)
    if any_one.any():
        ax.scatter(core_obs.loc[any_one, 'cx'], core_obs.loc[any_one, 'cy'],
                   s=dot_size_one_of, c=color_any_one, alpha=0.5,
                   linewidths=0, zorder=4, rasterized=True)
    if triple.any():
        ax.scatter(core_obs.loc[triple, 'cx'], core_obs.loc[triple, 'cy'],
                   s=dot_size_triple, c=triple_color, alpha=1.0,
                   linewidths=0, zorder=7, rasterized=True)

    ax.set_title("GDF15 ∩ Stemness ∩ SenePy", fontsize=11, fontweight='bold', color='black')
    _clean_ax(ax)

    ax = axes[1][3]
    _draw_bg(ax)

    gdf15_only    = is_epi & has_g & ~has_s & ~has_p
    stem_only     = is_epi & ~has_g & has_s & ~has_p
    sene_only     = is_epi & ~has_g & ~has_s & has_p
    gdf15_stem    = is_epi & has_g & has_s & ~has_p
    stem_sene     = is_epi & ~has_g & has_s & has_p
    gdf15_sene    = is_epi & has_g & ~has_s & has_p
    all_three     = is_epi & has_g & has_s & has_p
    none_epi      = is_epi & ~has_g & ~has_s & ~has_p

    layers = [
        (none_epi,     color_other_epi,                                    alpha_other_epi, dot_size_other,  3),
        (gdf15_only,   category_colors.get("GDF15 only", "#EA2027"),       0.8, dot_size_dense,  4),
        (stem_only,    category_colors.get("Stemness only", "#44c3e3"),    0.8, dot_size_dense,  4),
        (sene_only,    category_colors.get("SenePy only", "#F79F1F"),      0.8, dot_size_dense,  4),
        (gdf15_stem,   category_colors.get("GDF15 ∩ Stemness", "#5758BB"), 0.9, dot_size_both,   5),
        (stem_sene,    category_colors.get("Stemness ∩ SenePy", "#046e27"),0.9, dot_size_both,   5),
        (gdf15_sene,   category_colors.get("GDF15 ∩ SenePy", "#fc8a60"),   0.9, dot_size_both,   5),
        (all_three,    category_colors.get("All three", "#000000"),         1.0, dot_size_triple, 7),
    ]

    for mask, color, alpha, size, zorder in layers:
        if mask.any():
            ax.scatter(core_obs.loc[mask, 'cx'], core_obs.loc[mask, 'cy'],
                       s=size, c=color, alpha=alpha, linewidths=0, zorder=zorder,
                       rasterized=True)

    ax.set_title("All Categories", fontsize=11, fontweight='bold', color='black')
    _clean_ax(ax)

    fig.tight_layout()

    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(f"{save_dir}/core_{core_id}_with_he.pdf",
                    dpi=300, bbox_inches='tight', facecolor='white')

    plt.show()

    fig_legend, ax_legend = plt.subplots(figsize=(4, 5))
    ax_legend.axis('off')

    legend_order = [
        ("GDF15 only",                  category_colors.get("GDF15 only", "#EA2027")),
        ("Stemness only",               category_colors.get("Stemness only", "#44c3e3")),
        ("SenePy only",                 category_colors.get("SenePy only", "#F79F1F")),
        ("GDF15 ∩ Stemness",            category_colors.get("GDF15 ∩ Stemness", "#5758BB")),
        ("Stemness ∩ SenePy",           category_colors.get("Stemness ∩ SenePy", "#046e27")),
        ("GDF15 ∩ SenePy",              category_colors.get("GDF15 ∩ SenePy", "#fc8a60")),
        ("GDF15 ∩ Stemness ∩ SenePy",   category_colors.get("All three", "#000000")),
        ("Other Epithelial",            color_other_epi),
    ]

    handles = [
        plt.Line2D([0], [0], marker='s', color='w',
                   markerfacecolor=color, markersize=12, label=label)
        for label, color in legend_order
    ]

    ax_legend.legend(
        handles=handles,
        loc='center',
        fontsize=12,
        frameon=True,
        framealpha=0.9,
        handletextpad=1.0,
        labelspacing=1.0,
    )

    fig_legend.tight_layout()

    if save_dir:
        fig_legend.savefig(f"{save_dir}/core_legend.pdf",
                           dpi=300, bbox_inches='tight', facecolor='white')

    plt.show()


plot_dense_grid_core_with_he(
    filtered_spatialdata_dict,
    core_id="R1-TA-2",
    feature_colors={
        "GDF15_dense":                  "#EA2027",
        "stemness_score_dense":         "#44c3e3",
        "senepy_intestine_epi_0_dense": "#F79F1F",
    },
    overlap_pairs=[
        ("GDF15_dense", "stemness_score_dense",                  "#5758BB"),
        ("stemness_score_dense", "senepy_intestine_epi_0_dense", "#046e27"),
    ],
    triple_color="#000000",
    color_other_epi="#d3d3d3",
    color_non_epi="#fafafa",
    color_any_one="#d3d3d3",
    dot_size_dense=12,
    dot_size_triple=18,
    alpha_dense=1.0,
    category_colors=CATEGORY_COLORS,
    save_dir=output_dir,
)

In [ ]:
cluster_colors = {
    "0": '#009432',
    "1": '#C4E538',
    "2": "#FF788F",
    "3": "#85D5FB",
    "4": "#B3A1F9",
    "5": '#0652DD',
    "6": '#F79F1F',
    "7": "#a53707",
    "8": '#833471',
    "9": '#EA2027',
    "10": '#1B1464'
}


In [ ]:
def _classify_overlap(row):
    """Classify an epithelial cell into one of 8 categories (including 'None')."""
    g = row["_g"]
    s = row["_s"]
    p = row["_p"]
    if g and s and p:
        return "All three"
    elif g and s:
        return "GDF15 ∩ Stemness"
    elif g and p:
        return "GDF15 ∩ SenePy"
    elif s and p:
        return "Stemness ∩ SenePy"
    elif g:
        return "GDF15 only"
    elif s:
        return "Stemness only"
    elif p:
        return "SenePy only"
    else:
        return "None"

import matplotlib.patches as mpatches

In [ ]:
DENSE_FEATURES = [
    "GDF15_dense",
    "stemness_score_dense",
    "senepy_intestine_epi_0_dense",
]

FEATURE_LABELS = {
    "GDF15_dense":                  "GDF15",
    "stemness_score_dense":         "Stemness",
    "senepy_intestine_epi_0_dense": "SenePy",
}

FEATURE_COLORS = {
    "GDF15_dense":                  "#d62728",
    "stemness_score_dense":         "#1f78b4",
    "senepy_intestine_epi_0_dense": "#f0c929",
}

OVERLAP_PAIRS = [
    ("GDF15_dense", "stemness_score_dense",                   "#7b2d8e"),
    ("stemness_score_dense", "senepy_intestine_epi_0_dense",  "#2ca02c"),
]

TRIPLE_COLOR    = "#000000"
COLOR_OTHER_EPI = "#e0e0e0"
COLOR_NON_EPI   = "#f5f5f5"
ALPHA_DENSE     = 0.9
ALPHA_OTHER_EPI = 0.4
ALPHA_NON_EPI   = 0.15
ALPHA_BOTH      = 1.0
ALPHA_ONE_OF    = 0.7
DOT_SIZE_DENSE  = 8
DOT_SIZE_OTHER  = 2
DOT_SIZE_NON_EPI = 0.5
DOT_SIZE_BOTH   = 12
DOT_SIZE_ONE_OF = 6
DOT_SIZE_TRIPLE = 14
FIGSIZE_PER_SUBPLOT = (4.5, 4.5)

TISSUE_ORDER = ['Dist_N', 'Adj_N', 'AD', 'CA']

# overlap categories 
CATEGORY_ORDER = [
    "GDF15 only",
    "Stemness only",
    "SenePy only",
    "GDF15 ∩ Stemness",
    "Stemness ∩ SenePy",
    "GDF15 ∩ SenePy",
    "All three",
]


CATEGORY_COLORS = {
    "GDF15 only":         "#EA2027",
    "Stemness only":      "#44c3e3",
    "SenePy only":        "#F79F1F",
    "GDF15 ∩ Stemness":   "#5758BB",
    "Stemness ∩ SenePy":  "#046e27",
    "GDF15 ∩ SenePy":     "#EE5A24",  # orange for the pair not shown spatially
    "All three":          "#000000",
}


def plot_dense_stacked_barplot_by_tissue(
    all_cells_adata,
    dense_features=DENSE_FEATURES,
    feature_labels=FEATURE_LABELS,
    category_order=CATEGORY_ORDER,
    category_colors=CATEGORY_COLORS,
    tissue_order=TISSUE_ORDER,
    tissue_colors=None,
    epithelial_col="annotation_final_fine_cd8",
    epithelial_label="Epithelial",
    tissue_col="tissue_type_cell_level_normal_split",
    normalize=True,
    figsize=(6, 5),
    save_path=None,
):
    if tissue_colors is None:
        tissue_colors = {
            'Dist_N': '#2c963d',
            'Adj_N':          '#40407a',
            'AD':         '#ffb142',
            'CA':         '#b33939',
        }

    obs = all_cells_adata.obs.copy()
    epi = obs[obs[epithelial_col] == epithelial_label].copy()

    feat_a, feat_b, feat_c = dense_features
    epi['_g'] = epi[feat_a] == True if feat_a in epi.columns else False
    epi['_s'] = epi[feat_b] == True if feat_b in epi.columns else False
    epi['_p'] = epi[feat_c] == True if feat_c in epi.columns else False
    epi['_category'] = epi.apply(_classify_overlap, axis=1)

    epi = epi[epi['_category'] != 'None'].copy()

    counts = epi.groupby([tissue_col, '_category']).size().unstack(fill_value=0)
    for cat in category_order:
        if cat not in counts.columns:
            counts[cat] = 0
    counts = counts[category_order]
    counts = counts.reindex([t for t in tissue_order if t in counts.index])

    if normalize:
        row_totals = counts.sum(axis=1)
        counts = counts.div(row_totals, axis=0)

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(counts))
    bottom = np.zeros(len(counts))

    for cat in category_order:
        vals = counts[cat].values
        ax.bar(x, vals, bottom=bottom, color=category_colors[cat], label=cat,
               width=0.6, edgecolor='white', linewidth=0.3)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(counts.index, fontsize=10, fontweight='bold')
    ax.set_ylabel("Proportion of epithelial cells" if normalize else "Number of epithelial cells",
                   fontsize=10)

    handles = [mpatches.Patch(facecolor=category_colors[c], label=c) for c in category_order]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8, frameon=True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.tight_layout()
    if save_path:
        fig.savefig(f"{save_path}/morans_combo_barplot.pdf",
            dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

import matplotlib.patches as mpatches
import matplotlib.cm as cm


def plot_dense_stacked_barplots_by_category(
    all_cells_adata,
    dense_features=DENSE_FEATURES,
    feature_labels=FEATURE_LABELS,
    category_order=CATEGORY_ORDER,
    category_colors=CATEGORY_COLORS,
    tissue_order=TISSUE_ORDER,
    tissue_colors=None,
    patient_colors=None,
    epithelial_col="annotation_final_fine_cd8",
    epithelial_label="Epithelial",
    tissue_col="tissue_type_cell_level_normal_split",
    none_color="#e9f1f2",
    normalize=True,
    figsize=(18, 7),
    save_dir=None,
):
    import os

    if tissue_colors is None:
        tissue_colors = {
            'Dist_N': '#2c963d',
            'Adj_N':          '#40407a',
            'AD':         '#ffb142',
            'CA':         '#b33939',
        }

    full_category_order = category_order + ["None"]
    full_category_colors = {**category_colors, "None": none_color}

    tissue_rank = {t: i for i, t in enumerate(tissue_order)}

    obs = all_cells_adata.obs.copy()
    epi = obs[obs[epithelial_col] == epithelial_label].copy()

    feat_a, feat_b, feat_c = dense_features
    epi['_g'] = epi[feat_a] == True if feat_a in epi.columns else False
    epi['_s'] = epi[feat_b] == True if feat_b in epi.columns else False
    epi['_p'] = epi[feat_c] == True if feat_c in epi.columns else False
    epi['_category'] = epi.apply(_classify_overlap, axis=1)

    epi['_group'] = epi[tissue_col].astype(str) + " | " + epi['patient_id'].astype(str)
    group_tissue = epi.groupby('_group')[tissue_col].first().to_dict()
    group_patient = epi.groupby('_group')['patient_id'].first().astype(str).to_dict()

    unique_patients = sorted(epi['patient_id'].astype(str).unique())
    if patient_colors is None:
        unique_patients_full = sorted(all_cells_adata.obs['patient_id'].astype(str).unique())
        patient_colors = dict(zip(
            unique_patients_full,
            sns.color_palette('tab20', len(unique_patients_full))
        ))

    counts = epi.groupby(['_group', '_category']).size().unstack(fill_value=0)
    for cat in full_category_order:
        if cat not in counts.columns:
            counts[cat] = 0
    counts = counts[full_category_order]

    if normalize:
        row_totals = counts.sum(axis=1)
        counts_norm = counts.div(row_totals, axis=0)
    else:
        counts_norm = counts.copy()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for sort_cat in full_category_order:
        sorted_groups = counts_norm.sort_values(sort_cat, ascending=True).index
        plot_data = counts_norm.loc[sorted_groups]
        stack_order = [sort_cat] + [c for c in full_category_order if c != sort_cat]

        fig = plt.figure(figsize=figsize)
        gs = fig.add_gridspec(
            3, 2,
            height_ratios=[20, 1, 1],
            width_ratios=[1, 0.15],
            hspace=0.02,
            wspace=0.02,
        )

        ax = fig.add_subplot(gs[0, 0])
        ax_tissue = fig.add_subplot(gs[1, 0], sharex=ax)
        ax_patient = fig.add_subplot(gs[2, 0], sharex=ax)
        ax_legend = fig.add_subplot(gs[:, 1])
        ax_legend.axis('off')

        x = np.arange(len(plot_data))
        bottom = np.zeros(len(plot_data))

        for cat in stack_order:
            vals = plot_data[cat].values
            ax.bar(x, vals, bottom=bottom, color=full_category_colors[cat], label=cat,
                   width=0.85, edgecolor='white', linewidth=0.3)
            bottom += vals

        ax.set_ylabel(
            "Proportion of epithelial cells" if normalize else "Number of epithelial cells",
            fontsize=10,
        )
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(bottom=False, labelbottom=False)

        for i, group in enumerate(sorted_groups):
            tissue = group_tissue.get(group, 'unknown')
            tc = tissue_colors.get(tissue, '#9E9E9E')
            ax_tissue.bar(i, 1, width=0.85, color=tc, edgecolor='none')
        ax_tissue.set_yticks([])
        ax_tissue.set_ylabel("Tissue", fontsize=7, rotation=0, labelpad=35, va='center')
        for spine in ax_tissue.spines.values():
            spine.set_visible(False)
        ax_tissue.tick_params(bottom=False, labelbottom=False)

        for i, group in enumerate(sorted_groups):
            pid = group_patient.get(group, 'unknown')
            pc = patient_colors.get(pid, '#9E9E9E')
            ax_patient.bar(i, 1, width=0.85, color=pc, edgecolor='none')
        ax_patient.set_yticks([])
        ax_patient.set_ylabel("Patient", fontsize=7, rotation=0, labelpad=35, va='center')
        ax_patient.set_xticks([])
        for spine in ax_patient.spines.values():
            spine.set_visible(False)

        ax.set_xlim(-0.5, len(plot_data) - 0.5)
        ax_tissue.set_xlim(-0.5, len(plot_data) - 0.5)
        ax_patient.set_xlim(-0.5, len(plot_data) - 0.5)

        cat_handles = [mpatches.Patch(facecolor=full_category_colors[c], label=c) for c in stack_order]
        tissue_handles = [mpatches.Patch(facecolor=tissue_colors[t], label=t)
                          for t in tissue_order if t in tissue_colors]
        patient_handles = [mpatches.Patch(facecolor=patient_colors[p], label=p)
                           for p in unique_patients]

        leg1 = ax_legend.legend(
            handles=cat_handles, loc='upper left', title="Category",
            fontsize=6, title_fontsize=7, frameon=True, framealpha=0.9,
            bbox_to_anchor=(0, 0.98),
        )
        ax_legend.add_artist(leg1)

        leg2 = ax_legend.legend(
            handles=tissue_handles, loc='upper left', title="Tissue",
            fontsize=6, title_fontsize=7, frameon=True, framealpha=0.9,
            bbox_to_anchor=(0, 0.72),
        )
        ax_legend.add_artist(leg2)

        ax_legend.legend(
            handles=patient_handles, loc='upper left', title="Patient",
            fontsize=5, title_fontsize=7, frameon=True, framealpha=0.9,
            bbox_to_anchor=(0, 0.56),
        )

        fig.tight_layout()

        if save_dir:
            safe_name = sort_cat.replace(" ", "_").replace("∩", "x")
            fig.savefig(
                f"{save_dir}/barplot_sorted_by_{safe_name}.png",
                dpi=150, bbox_inches='tight', facecolor='white',
            )

        plt.show()

    print("All barplots done.")

In [ ]:
plot_dense_stacked_barplot_by_tissue(
    all_cells_adata, 
    save_path = output_dir
)

In [ ]:
obs = all_cells_adata.obs.copy()
epi = obs[obs['annotation_final_fine_cd8'] == 'Epithelial'].copy()

feat_a, feat_b, feat_c = DENSE_FEATURES
epi['_g'] = epi[feat_a] == True
epi['_s'] = epi[feat_b] == True
epi['_p'] = epi[feat_c] == True
epi['_category'] = epi.apply(_classify_overlap, axis=1)

print(epi['_category'].value_counts())
print()
print(epi[['_g', '_s', '_p']].sum())
print()
print(f"Total epithelial cells: {len(epi)}")
print(f"None cells: {(epi['_category'] == 'None').sum()}")
print()
print(epi[feat_a].dtype, epi[feat_a].value_counts().head())

In [ ]:
def plot_dense_stacked_barplot_single(
    all_cells_adata,
    sort_by,
    dense_features=DENSE_FEATURES,
    feature_labels=FEATURE_LABELS,
    category_order=CATEGORY_ORDER,
    category_colors=CATEGORY_COLORS,
    tissue_order=TISSUE_ORDER,
    tissue_colors=tissue_colors_normal_split,
    patient_colors=None,
    epithelial_col="annotation_final_fine_cd8",
    epithelial_label="Epithelial",
    tissue_col="tissue_type_cell_level_normal_split",
    none_color="#ebf2fa",
    normalize=True,
    figsize=(18, 7),
    save_dir=None,
):
    import os

    if tissue_colors is None:
        tissue_colors = {
            'Dist_N': '#2c963d',
            'Adj_N':          '#40407a',
            'AD':         '#ffb142',
            'CA':         '#b33939',
        }

    full_category_order = category_order + ["None"]
    full_category_colors = {**category_colors, "None": none_color}

    obs = all_cells_adata.obs.copy()
    epi = obs[obs[epithelial_col] == epithelial_label].copy()

    feat_a, feat_b, feat_c = dense_features
    epi['_g'] = epi[feat_a] == True if feat_a in epi.columns else False
    epi['_s'] = epi[feat_b] == True if feat_b in epi.columns else False
    epi['_p'] = epi[feat_c] == True if feat_c in epi.columns else False
    epi['_category'] = epi.apply(_classify_overlap, axis=1)

    epi['_group'] = epi[tissue_col].astype(str) + " | " + epi['patient_id'].astype(str)
    group_tissue = epi.groupby('_group')[tissue_col].first().to_dict()
    group_patient = epi.groupby('_group')['patient_id'].first().astype(str).to_dict()

    unique_patients = sorted(epi['patient_id'].astype(str).unique())
    if patient_colors is None:
        unique_patients_full = sorted(all_cells_adata.obs['patient_id'].astype(str).unique())
        patient_colors = dict(zip(
            unique_patients_full,
            sns.color_palette('tab20', len(unique_patients_full))
        ))

    counts = epi.groupby(['_group', '_category']).size().unstack(fill_value=0)
    for cat in full_category_order:
        if cat not in counts.columns:
            counts[cat] = 0
    counts = counts[full_category_order]

    if normalize:
        row_totals = counts.sum(axis=1)
        counts_norm = counts.div(row_totals, axis=0)
    else:
        counts_norm = counts.copy()

    sorted_groups = counts_norm.sort_values(sort_by, ascending=True).index
    plot_data = counts_norm.loc[sorted_groups]
    stack_order = [sort_by] + [c for c in full_category_order if c != sort_by]

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        3, 2,
        height_ratios=[20, 1, 1],
        width_ratios=[1, 0.15],
        hspace=0.02,
        wspace=0.02,
    )

    ax = fig.add_subplot(gs[0, 0])
    ax_tissue = fig.add_subplot(gs[1, 0], sharex=ax)
    ax_patient = fig.add_subplot(gs[2, 0], sharex=ax)
    ax_legend = fig.add_subplot(gs[:, 1])
    ax_legend.axis('off')

    x = np.arange(len(plot_data))
    bottom = np.zeros(len(plot_data))

    for cat in stack_order:
        vals = plot_data[cat].values
        ax.bar(x, vals, bottom=bottom, color=full_category_colors[cat], label=cat,
               width=0.85, edgecolor='white', linewidth=0.3)
        bottom += vals

    ax.set_ylabel(
        "Proportion of epithelial cells" if normalize else "Number of epithelial cells",
        fontsize=14, fontweight='bold', labelpad=20,
    )
    ax.tick_params(axis='y', labelsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(bottom=False, labelbottom=False)

    for i, group in enumerate(sorted_groups):
        tissue = group_tissue.get(group, 'unknown')
        tc = tissue_colors.get(tissue, '#9E9E9E')
        ax_tissue.bar(i, 1, width=0.85, color=tc, edgecolor='none')
    ax_tissue.set_yticks([])
    ax_tissue.set_ylabel("Tissue", fontsize=12, fontweight='bold', rotation=0, labelpad=60, va='center')
    for spine in ax_tissue.spines.values():
        spine.set_visible(False)
    ax_tissue.tick_params(bottom=False, labelbottom=False)

    for i, group in enumerate(sorted_groups):
        pid = group_patient.get(group, 'unknown')
        pc = patient_colors.get(pid, '#9E9E9E')
        ax_patient.bar(i, 1, width=0.85, color=pc, edgecolor='none')
    ax_patient.set_yticks([])
    ax_patient.set_ylabel("Patient", fontsize=12, fontweight='bold', rotation=0, labelpad=60, va='center')
    ax_patient.set_xticks([])
    for spine in ax_patient.spines.values():
        spine.set_visible(False)

    ax.set_xlim(-0.5, len(plot_data) - 0.5)
    ax_tissue.set_xlim(-0.5, len(plot_data) - 0.5)
    ax_patient.set_xlim(-0.5, len(plot_data) - 0.5)

    cat_handles = [mpatches.Patch(facecolor=full_category_colors[c], label=c) for c in stack_order]
    tissue_handles = [mpatches.Patch(facecolor=tissue_colors[t], label=t)
                      for t in tissue_order if t in tissue_colors]
    patient_handles = [mpatches.Patch(facecolor=patient_colors[p], label=p)
                       for p in unique_patients]

    leg1 = ax_legend.legend(
        handles=cat_handles, loc='upper left', title="Category",
        fontsize=7, title_fontsize=10, frameon=True, framealpha=0.9,
        bbox_to_anchor=(0, 0.98),
    )
    ax_legend.add_artist(leg1)

    leg2 = ax_legend.legend(
        handles=tissue_handles, loc='upper left', title="Tissue",
        fontsize=7, title_fontsize=10, frameon=True, framealpha=0.9,
        bbox_to_anchor=(0, 0.72),
    )
    ax_legend.add_artist(leg2)

    ax_legend.legend(
        handles=patient_handles, loc='upper left', title="Patient",
        fontsize=7, title_fontsize=10, frameon=True, framealpha=0.9,
        bbox_to_anchor=(0, 0.56), ncols=2
    )

    fig.tight_layout()

    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        safe_name = sort_by.replace(" ", "_").replace("∩", "x")
        fig.savefig(f"{save_dir}/morans_barplot.pdf",
            dpi=300, bbox_inches='tight', facecolor='white')

    plt.show()

In [ ]:
plot_dense_stacked_barplot_single(
    all_cells_adata,
    sort_by="Stemness ∩ SenePy",
    save_dir = output_dir,
    patient_colors=patient_colors_shared
)


# Overlap category violin plots

In [ ]:
# ── Compute overlap categories from all_cells_adata ──────────────────────────

DENSE_FEATURES_OC = ["GDF15_dense", "stemness_score_dense", "senepy_intestine_epi_0_dense"]


epi_mask = all_cells_adata.obs["annotation_final_fine_cd8"] == "Epithelial"
epi_obs = all_cells_adata.obs.loc[epi_mask].copy()

for feat in DENSE_FEATURES_OC:
    epi_obs[feat] = epi_obs[feat].fillna(False).astype(bool)

has_gdf15  = epi_obs["GDF15_dense"]
has_stem   = epi_obs["stemness_score_dense"]
has_senepy = epi_obs["senepy_intestine_epi_0_dense"]


conditions = []
labels = []

conditions.append(has_gdf15 & has_stem & has_senepy)
labels.append("GDF15+Stem+SenePy")

conditions.append(has_gdf15 & has_stem & ~has_senepy)
labels.append("GDF15+Stem")

conditions.append(has_gdf15 & ~has_stem & has_senepy)
labels.append("GDF15+SenePy")

conditions.append(~has_gdf15 & has_stem & has_senepy)
labels.append("Stem+SenePy")

conditions.append(has_gdf15 & ~has_stem & ~has_senepy)
labels.append("GDF15_only")

conditions.append(~has_gdf15 & has_stem & ~has_senepy)
labels.append("Stem_only")

conditions.append(~has_gdf15 & ~has_stem & has_senepy)
labels.append("SenePy_only")

conditions.append(~has_gdf15 & ~has_stem & ~has_senepy)
labels.append("None")

overlap_cat = pd.Series("None", index=epi_obs.index, dtype="object")
for cond, label in zip(conditions, labels):
    overlap_cat.loc[cond] = label

epi_obs["overlap_category"] = pd.Categorical(overlap_cat, categories=labels, ordered=False)

print("Overlap category counts (epithelial cells in all_cells_adata):")
print(epi_obs["overlap_category"].value_counts().to_string())
print(f"\nTotal epithelial cells classified: {len(epi_obs)}")

In [ ]:

overlap_map = epi_obs.set_index("cell_id")["overlap_category"]

epi_adata.obs["overlap_category"] = (
    epi_adata.obs["cell_id"]
    .map(overlap_map)
    .astype("category")
)

n_mapped = epi_adata.obs["overlap_category"].notna().sum()
n_total = len(epi_adata)
print(f"Mapped {n_mapped}/{n_total} cells in epi_adata ({100*n_mapped/n_total:.1f}%)")
print(epi_adata.obs["overlap_category"].value_counts())

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 1, figsize=(3, 5))

group_order = ["Stem_only", "SenePy_only", "Stem+SenePy"]
group_colors = ["#44c3e3", "#F79F1F", "#046e27"]
tissue_col = "tissue_type_cell_level_normal_split"

mask = (
    epi_adata.obs[tissue_col].eq("AD")
    & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
)
adata_sub = epi_adata[mask].copy()
adata_sub.obs["deg_group"] = pd.Categorical(
    adata_sub.obs["overlap_category"].astype(str),
    categories=group_order,
    ordered=True,
)

for ax, gene in zip(axes, ["OLFM4", "APCDD1"]):
    gene_expr = adata_sub[:, gene].X.toarray().flatten() if hasattr(adata_sub[:, gene].X, "toarray") else np.asarray(adata_sub[:, gene].X).flatten()
    plot_df = pd.DataFrame({"expression": gene_expr, "group": adata_sub.obs["deg_group"].values})

    sns.violinplot(
        data=plot_df,
        y="group",
        x="expression",
        order=group_order,
        palette=group_colors,
        orient="h",
        inner=None,
        cut=0,
        ax=ax,
    )
    ax.set_title(f"AD: {gene}", fontsize=12)
    ax.set_ylabel("")
    ax.set_xlabel(gene)

plt.tight_layout()
fig.savefig(os.path.join(output_dir, "violin_TA_OLFM4_APCDD1_horizontal.pdf"),
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()